[Reference](https://medium.com/@Rohan_Dutt/10-functional-data-processing-techniques-every-data-analyst-must-deploy-in-production-1e1b70ec3eb7)

# 1. Optimize with Lazy Evaluation

In [1]:
import itertools
import pandas as pd
from typing import Iterator, Dict

# Traditional eager loading (loads 10M rows into memory)
# df = pd.read_csv('massive_file.csv')  # Memory crash imminent

# Lazy generator approach (constant memory)
def gen_filtered_records(filepath: str, min_value: float) -> Iterator[Dict]:
    """
    Streams rows, filters on the fly, memory usage stays flat.
    Perfect for 100GB+ log files.
    """
    for chunk in pd.read_csv(filepath, chunksize=100_000):
        # Process chunk without loading full file
        filtered = chunk[chunk['revenue'] > min_value]
        for _, row in filtered.iterrows():
            yield row.to_dict()

# Chain multiple lazy operations
def calculate_metrics(records: Iterator[Dict]) -> Iterator[Dict]:
    """Pure transformation, nothing executed yet"""
    for record in records:
        record['profit_margin'] = (record['revenue'] - record['cost']) / record['revenue']
        yield record

# Only when you iterate does anything compute
pipeline = calculate_metrics(gen_filtered_records('sales.csv', min_value=1000))

# Process results incrementally (e.g., write to database)
for i, result in enumerate(pipeline):
    if i >= 1000:  # Stop early without processing entire file
        break
    # db.insert(result)  # Real insertion here

# 2. Chain Higher-Order Functions Like a Pro

In [2]:
from functools import reduce
from itertools import filterfalse
from operator import itemgetter
import toolz

# Raw data: messy API responses
transactions = [
    {'id': 1, 'amount': 150.0, 'status': 'approved'},
    {'id': 2, 'amount': 0.0, 'status': 'pending'},
    {'id': 3, 'amount': -50.0, 'status': 'approved'},
    {'id': 4, 'amount': 200.0, 'status': 'rejected'}
]

# PRO: Clean chaining (no intermediate variables)
valid_total = (
    transactions
    |> toolz.curry(filter)(lambda x: x['status'] == 'approved')
    |> toolz.curry(filter)(lambda x: x['amount'] > 0)
    |> toolz.curry(map)(itemgetter('amount'))
    |> sum
)

# With reduce: Complex aggregation in one line
def running_average(acc, val):
    """Stateful aggregation without mutating external variables"""
    total, count = acc
    return (total + val, count + 1)

result = reduce(running_average, [10, 20, 30, 40], (0, 0))
average = result[0] / result[1]  # 25.0

# Production pattern: Compose reusable validators
validators = [
    lambda rec: rec.get('amount', 0) > 0,
    lambda rec: rec.get('status') == 'approved',
    lambda rec: 'user_id' in rec
]

# Apply all validators lazily
valid_records = filter(
    lambda rec: all(v(rec) for v in validators),
    transactions
)

# 3. Master Data Partitioning for Parallel Processing
```
Use Dask for automatic partitioning or manual multiprocessing for custom logic.
```

# 4. Leverage Memoization to Cache Costly Operations


In [3]:
from functools import lru_cache
import pandas as pd
import hashlib
import pickle
import os

# Cache expensive geocoding API calls
@lru_cache(maxsize=1024)
def geocode_address(address: str) -> tuple:
    """
    LRU cache prevents duplicate API calls.
    1024 most recent addresses cached in memory.
    """
    # Simulate expensive API call
    import time; time.sleep(0.1)  # Real API latency
    lat, lng = hash(address) % 90, hash(address) % 180
    return (lat, lng)

# Production-grade persistent cache for pandas operations
class PersistentCache:
    def __init__(self, cache_dir='./cache'):
        os.makedirs(cache_dir, exist_ok=True)
        self.cache_dir = cache_dir

    def _hash_args(self, func_name, args) -> str:
        """Create unique key from args"""
        key_str = f"{func_name}_{str(args)}"
        return hashlib.md5(key_str.encode()).hexdigest()

    def __call__(self, func):
        def wrapper(*args, kwargs):
            cache_key = self._hash_args(func.__name__, args)
            cache_path = os.path.join(self.cache_dir, f"{cache_key}.pkl")

            if os.path.exists(cache_path):
                with open(cache_path, 'rb') as f:
                    return pickle.load(f)

            result = func(*args, kwargs)
            with open(cache_path, 'wb') as f:
                pickle.dump(result, f)
            return result
        return wrapper

@PersistentCache(cache_dir='./feature_cache')
def calculate_customer_lifetime_value(customer_id: int, df: pd.DataFrame) -> float:
    """Cache feature calculations across notebook reruns"""
    customer_orders = df[df['customer_id'] == customer_id]
    return (customer_orders['order_value'] * customer_orders['frequency']).sum()

# Usage in pandas apply (cache saves 90%+ compute)
df['clv'] = df['customer_id'].apply(
    lambda cid: calculate_customer_lifetime_value(cid, orders_df)
)

# 5. Use Pure Functions to Debug like a Scientist

In [4]:
import pandas as pd
from typing import Dict, Any
import copy

# IMPURE: Modifies original DataFrame (debugging nightmare)
def add_margin_impure(df: pd.DataFrame) -> None:
    df['margin'] = df['revenue'] - df['cost']  # Silent mutation

# PURE: Returns new DataFrame, original untouched
def add_margin_pure(df: pd.DataFrame) -> pd.DataFrame:
    """
    Immutable transformation enables:
    - Unit testing without fixtures
    - Parallel processing without race conditions
    - Time-travel debugging
    """
    return df.assign(margin=df['revenue'] - df['cost'])

# Production pattern: Pure data validation pipeline
def validate_record(record: Dict[str, Any]) -> Dict[str, Any]:
    """Pure function: no logging, no global state, no mutations"""
    if not (0 <= record.get('discount', 0) <= 1):
        raise ValueError(f"Invalid discount: {record['discount']}")
    return record  # Return validated copy

def enrich_record(record: Dict[str, Any]) -> Dict[str, Any]:
    """Pure transformation"""
    validated = validate_record(record)
    return {
        validated,
        'net_price': validated['price'] * (1 - validated.get('discount', 0))
    }

# usage in pipeline: Thread-safe and testable
raw_records = [{'price': 100, 'discount': 0.2}, {'price': 50, 'discount': 0.1}]
enriched = list(map(enrich_record, raw_records))

# Pandas immutability pattern
df_new = (df
    .query("status == 'active'")  # Returns new df
    .pipe(lambda x: x.assign(ratio=x['a']/x['b']))  # Pure pipe
    .reset_index(drop=True)  # No side effects
)

# 6. Exploit Vectorized Operations over Loops


In [5]:
import pandas as pd
import numpy as np
from numba import jit
import time

# ANTI-PATTERN: Python loops (0.5M rows = 2.3 seconds)
def calculate_rolling_mean_loop(df: pd.DataFrame, window: int) -> pd.Series:
    result = []
    for i in range(len(df)):
        if i < window - 1:
            result.append(np.nan)
        else:
            result.append(df['value'].iloc[i-window+1:i+1].mean())
    return pd.Series(result)

# VECTORIZED: Numpy operations (15 milliseconds = 150x faster)
def calculate_rolling_mean_vectorized(df: pd.DataFrame, window: int) -> pd.Series:
    return df['value'].rolling(window).mean()

# EXTREME: Numba JIT for complex logic that can't be vectorized
@jit(nopython=True)  # Compiles to machine code
def custom_signal_jit(values: np.ndarray, threshold: float) -> np.ndarray:
    """
    Loops compiled to SIMD instructions.
    50x faster than pure Python for arbitrary logic.
    """
    signals = np.empty_like(values)
    for i in range(len(values)):
        if i == 0:
            signals[i] = 0
        elif values[i] > threshold and values[i-1] <= threshold:
            signals[i] = 1
        else:
            signals[i] = 0
    return signals

# Benchmark setup
df = pd.DataFrame({'value': np.random.randn(500_000)})

# Measure loop time
start = time.time()
_ = calculate_rolling_mean_loop(df, 20)
loop_time = time.time() - start

# Measure vectorized time
start = time.time()
_ = calculate_rolling_mean_vectorized(df, 20)
vec_time = time.time() - start

print(f"Loop: {loop_time:.3f}s, Vectorized: {vec_time:.3f}s, Speedup: {loop_time/vec_time:.0f}x")

# 7. Automate Schema Enforcement Early


In [6]:
import pandas as pd
import pandera as pa
from pandera import Column, Check
from pydantic import BaseModel, validator
from typing import List

# Pydantic: Enforce schema at API boundary
class Transaction(BaseModel):
    transaction_id: int
    amount: float
    status: str

    @validator('amount')
    def amount_positive(cls, v):
        if v <= 0:
            raise ValueError(f'Amount must be positive, got {v}')
        return v

    @validator('status')
    def status_valid(cls, v):
        if v not in ['approved', 'pending', 'rejected']:
            raise ValueError(f'Invalid status: {v}')
        return v

# Pandera: Runtime DataFrame validation
schema = pa.DataFrameSchema({
    "customer_id": Column(int, checks=Check.greater_than(0)),
    "order_date": Column(pa.DateTime),
    "order_value": Column(float, checks=[
        Check.greater_than(0),
        Check.less_than_or_equal_to(1_000_000)
    ]),
    "discount_rate": Column(float, nullable=True, checks=Check.between(0, 1))
})

def process_orders(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Schema validation fails fast with row-level error reporting.
    Prevents silent NaN propagation.
    """
    try:
        # Validate entire DataFrame
        validated_df = schema(raw_df)

        # Transformation logic only runs if validation passes
        return validated_df.assign(
            net_value=lambda df: df['order_value'] * (1 - df['discount_rate'].fillna(0))
        )
    except pa.errors.SchemaError as e:
        # Log specific failure row and column
        print(f"Schema violation at row {e.failure_cases.index.tolist()}: {e}")
        raise

# Production pattern: Validation as decorator
def validate_output(schema: pa.DataFrameSchema):
    def decorator(func):
        def wrapper(*args, kwargs):
            result = func(*args, kwargs)
            return schema(result)  # Auto-validate
        return wrapper
    return decorator

@validate_output(schema)
def ingest_from_api() -> pd.DataFrame:
    """Every return is automatically validated"""
    return pd.DataFrame({
        'customer_id': [1, 2],
        'order_date': pd.to_datetime(['2024-01-01', '2024-01-02']),
        'order_value': [100.0, 200.0],
        'discount_rate': [0.1, None]
    })

# 8. Stream Data, Don’t Load It All


In [7]:
import pandas as pd
import vaex
import sqlite3
from typing import Iterator

# PATTERN 1: Chunked processing with pandas
def stream_csv_to_db(csv_path: str, db_path: str, chunk_size: int = 500_000):
    """
    Process million-row chunks without ever loading full file.
    Constant memory ~500MB regardless of file size.
    """
    conn = sqlite3.connect(db_path)

    for i, chunk in enumerate(pd.read_csv(csv_path, chunksize=chunk_size)):
        # Clean and transform chunk
        chunk['processed_at'] = pd.Timestamp.now()
        chunk = chunk.dropna(subset=['user_id'])

        # Append to database, release memory
        chunk.to_sql('events', conn, if_exists='append', index=False)
        print(f"Chunk {i}: {len(chunk)} rows processed")

    conn.close()

# PATTERN 2: Vaex for huge files (zero memory overhead)
def analyze_with_vaex(hdf5_path: str):
    """
    Vaex memory-maps files; operations are lazy expressions.
    Sorts 100GB file on disk, not in RAM.
    """
    df = vaex.open(hdf5_path)

    # These create expression trees; no computation yet
    filtered = df[(df.transaction_amount > 1000) & (df.status == 'completed')]
    aggregated = filtered.groupby('customer_id', agg={'total_spend': 'sum'})

    # Only here does execution happen
    result = aggregated.sort('total_spend', ascending=False)
    return result.head(10).to_pandas_df()  # Materialize only top 10

# PATTERN 3: Generator pipeline for model training
def stream_features(file_pattern: str) -> Iterator[dict]:
    """
    Infinite data stream for online learning.
    Model trains while data flows; no memory limit.
    """
    import glob
    for filepath in glob.glob(file_pattern):
        for chunk in pd.read_parquet(filepath):
            # Feature engineering on the fly
            features = {
                'avg_session_time': chunk['session_duration'].mean(),
                'bounce_rate': (chunk['page_views'] == 1).mean(),
                'target': chunk['converted'].iloc[0]
            }
            yield features

# Usage in online learning
# for features in stream_features('daily_*.parquet'):
#     model.partial_fit([features], [features.pop('target')])

# 9. Compress Intermediate Data States


In [8]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.feather as feather
import os

def compare_formats(df: pd.DataFrame, filename_prefix: str):
    """
    Benchmark: Write same data to multiple formats.
    Shows real compression ratios and read times.
    """
    # CSV: 1.2GB for 10M rows
    csv_path = f"{filename_prefix}.csv"
    df.to_csv(csv_path, index=False)
    csv_size = os.path.getsize(csv_path)

    # Parquet: Best for long-term storage
    parquet_path = f"{filename_prefix}.parquet"
    df.to_parquet(
        parquet_path,
        compression='snappy',  # Good balance: fast + decent compression
        engine='pyarrow',
        index=False
    )
    parquet_size = os.path.getsize(parquet_path)

    # Feather: Fastest for short-term caching (same machine)
    feather_path = f"{filename_prefix}.feather"
    feather.write_feather(df, feather_path)
    feather_size = os.path.getsize(feather_path)

    print(f"CSV: {csv_size:,} bytes")
    print(f"Parquet: {parquet_size:,} bytes ({100*parquet_size/csv_size:.1f}%)")
    print(f"Feather: {feather_size:,} bytes ({100*feather_size/csv_size:.1f}%)")

    # Read speed comparison
    %timeit pd.read_csv(csv_path)  # 12.3s
    %timeit pd.read_parquet(parquet_path)  # 1.8s
    %timeit feather.read_feather(feather_path)  # 0.9s

# Production pattern: Incremental Parquet writes
def write_partitioned_parquet(df: pd.DataFrame, base_path: str, partition_cols: list):
    """
    Writes DataFrame partitioned by column values.
    Enables partition pruning in downstream queries.
    """
    df.to_parquet(
        base_path,
        engine='pyarrow',
        compression='snappy',
        partition_cols=partition_cols,
        index=False
    )
    # Creates structure: base_path/year=2024/month=01/data.parquet

# Reading with predicate pushdown
def read_recent_partitions(base_path: str) -> pd.DataFrame:
    """
    Only reads partitions matching filter; skips 99% of files.
    """
    return pd.read_parquet(
        base_path,
        engine='pyarrow',
        filters=[('year', '>=', 2024), ('month', '>=', 10)]
    )

# 10. Profile Before Optimizing (Measure Twice, Cut Once)


In [9]:
import cProfile
import pstats
from line_profiler import LineProfiler
import pandas as pd
import numpy as np

def pipeline_function():
    """Example pipeline to profile"""
    df = pd.DataFrame({
        'a': np.random.randn(1_000_000),
        'b': np.random.randn(1_000_000),
        'c': np.random.randn(1_000_000)
    })

    # Slow operations
    df['d'] = df.apply(lambda row: row['a'] + row['b'] * row['c'], axis=1)
    df['e'] = df['a'].rolling(100).apply(lambda x: np.percentile(x, 95))

# METHOD 1: cProfile for overall bottlenecks
profiler = cProfile.Profile()
profiler.enable()
pipeline_function()
profiler.disable()

# Print sorted stats
stats = pstats.Stats(profiler)
stats.sort_stats('cumulative')
stats.print_stats(10)  # Top 10 slowest functions

# METHOD 2: line_profiler for exact line hotspots
lp = LineProfiler()

@lp.add_function
def optimized_pipeline():
    df = pd.DataFrame({
        'a': np.random.randn(1_000_000),
        'b': pd.Series(np.random.randn(1_000_000)),
        'c': np.random.randn(1_000_000)
    })

    # Optimized: Vectorized replace apply
    df['d'] = df['a'] + df['b'] * df['c']

    # Optimized: Use rolling.quantile (Cython backend)
    df['e'] = df['a'].rolling(100).quantile(0.95)

lp.run('optimized_pipeline()')
lp.print_stats(output_unit=1e-3)  # Times in milliseconds

# METHOD 3: Memory profiling
from memory_profiler import profile

@profile
def memory_intensive_operation():
    """Shows line-by-line memory consumption"""
    df = pd.read_csv('large_file.csv')  # +500MB here
    filtered = df[df['value'] > 0]  # +250MB (copy)
    aggregated = filtered.groupby('id').sum()  # +100MB
    return aggregated

# Production pattern: Profile in CI/CD
def test_performance_regression():
    """
    Fail CI if pipeline runs >10% slower than baseline.
    """
    import time
    baseline_time = 8.5  # seconds

    start = time.time()
    pipeline_function()
    actual_time = time.time() - start

    assert actual_time < baseline_time * 1.1, \
        f"Performance regression: {actual_time:.1f}s > {baseline_time*1.1:.1f}s"